# Assigning AAL3 and Brodmann Atlas Labels to Brain Surfaces

This notebook assigns AAL3 and Brodmann atlas labels to the Colin27 and ICBM152 brain surfaces, visualizes the labels, and summarizes their overlap with the Schaefer2018 parcels.

## Overview

Neuroimaging atlases — parcellations that divide the cerebral cortex into anatomically
or functionally defined regions — are a cornerstone of group-level fNIRS and DOT
analysis. Atlases allow researchers to:

* Report results in a standardised anatomical vocabulary that is comparable across
  studies and laboratories.
* Aggregate channel or vertex signals into region-of-interest (ROI) time series,
  reducing the multiple-comparisons burden.
* Cross-reference DOT activations with the large fMRI literature, where the same
  atlas labels are widely used.

### MNI space as a shared coordinate system

Most brain atlases are defined and distributed in **MNI (Montreal Neurological
Institute) space**: a standardised brain coordinate frame derived from averaging many
individual MRI scans into a common template.
Every standard head model in Cedalion — Colin27
<cite data-cite="Holmes1998">(Holmes et al., 1998)</cite>
and ICBM152
<cite data-cite="Fonov2011">(Fonov et al., 2011)</cite> — carries pre-computed MNI152
coordinates for every brain surface vertex. This means that **any** volumetric atlas
provided as a NIfTI file in MNI space can be transferred to the brain surface without
any additional registration step: the shared coordinate system acts as the bridge
between the volumetric atlas and the surface model.

### The two MNI spaces: MNI152 and MNI305

Two MNI variants are in common use and it is important to keep them distinct:

| Space | Template | Typical users |
|---|---|---|
| **MNI152** | Average of 152 adult brains (ICBM152 template) | FSL, recent SPM, most modern atlases |
| **MNI305** | Average of 305 brains; the original MNI reference | FreeSurfer, older SPM, some legacy atlases |

MNI152 and MNI305 differ by a small affine shift (a few millimetres in some regions)
and are **not** interchangeable. Cedalion stores a pre-computed affine transform
(`cedalion.dot.utils.mni305_to_mni152`) and applies it automatically when a NIfTI is
declared as `voxel_label_crs='mni305'`, so you do not need to handle the conversion
manually. Both spaces are fully supported; you simply specify which one your NIfTI
file uses.

### What this notebook demonstrates

This notebook shows the general workflow for applying **any** user-supplied parcellation
scheme — provided as a NIfTI label volume in either MNI152 or MNI305 space — to the
Cedalion head models. The workflow is illustrated with two widely used atlases:

* **AAL3** (Automated Anatomical Labelling atlas, version 3)
  <cite data-cite="Rolls2020">(Rolls et al., 2020)</cite> — 170 macro-anatomical regions
  covering the entire cerebral cortex and subcortex.
* **Brodmann areas** (Mai–Majtanik atlas)
  <cite data-cite="Mai2017">(Mai & Majtanik, 2017)</cite> — the classical cytoarchitectonic
  map, available as a volumetric MNI label file.

Both atlases are bundled with Cedalion and loaded via `cedalion.data.get_atlas_files()`.
The same code applies unchanged to any other NIfTI atlas you supply.

**For more detail, see also:**
- [43a_head_models_overview.ipynb](43a_head_models_overview.ipynb) — introduction to
  `TwoSurfaceHeadModel` and the Schaefer2018 parcellation bundled with the standard models
- [43b_individualized_head_models.ipynb](43b_individualized_head_models.ipynb) — building an
  individualized head model from a subject's own MRI using FreeSurfer, which yields
  surface-native parcel labels with sharper boundaries than the volumetric approach shown here

In [ ]:
import json

import numpy as np
import pyvista as pv
from scipy.spatial import KDTree

import cedalion
import cedalion.dot
from cedalion.vis.anatomy import get_vertex_colors_from_coord, plot_brain_views_grid

pv.set_jupyter_backend("static")

## Loading Colin27 and ICBM152 Head Models

We load both standard atlas head models. Each is stored in voxel space (`crs='ijk'`);
`assign_parcels_via_mni_coords` works in MNI152 space internally so the coordinate
system of the loaded head model does not need to match that of the atlas.

Both models carry pre-computed `mni152_r/a/s` vertex coordinates on the brain surface,
which is what makes the atlas transfer possible. The inflated cortex surfaces are loaded
for visualization: inflation removes sulci and gyri so that buried cortex becomes
visible, making parcel boundaries much easier to inspect.

In [ ]:
colin_ijk = cedalion.dot.get_standard_headmodel("colin27")
colin_inflated = cedalion.dot.get_inflated_cortex_surface("colin27")

icbm_ijk = cedalion.dot.get_standard_headmodel("icbm152")
icbm_inflated = cedalion.dot.get_inflated_cortex_surface("icbm152")

## Loading Atlases

Atlases distributed for use with standard MNI templates typically come in two files:

1. A **NIfTI volume** (`.nii` or `.nii.gz`) in which every voxel inside a labelled
   region carries a positive integer, and background voxels carry 0 (or another
   reserved value).
2. A **label mapping** (`.json`, `.csv`, or `.txt`) that translates those integers to
   human-readable region names.

`cedalion.data.get_atlas_files(name)` returns both paths for the bundled atlases.
For a custom atlas you can pass any NIfTI path you have downloaded or created.

### AAL3 — Automated Anatomical Labelling atlas (version 3)

AAL3 <cite data-cite="Rolls2020">(Rolls et al., 2020)</cite> divides the cerebral cortex
and subcortical structures into 170 macro-anatomical regions based on sulcal and gyral
landmarks visible in the MNI152 template. It is one of the most widely used atlases in
the fMRI and fNIRS literature and provides a common vocabulary for reporting results.
The NIfTI file bundled with Cedalion is defined in **MNI152** space.

In [ ]:
aal3_voxel_label_niftii, aal3_labels_json = cedalion.data.get_atlas_files("aal3")

# dictionary to map numeric voxel labels in nifti to string labels
with aal3_labels_json.open("r") as fin:
    aal3_num2label = json.load(fin)
    aal3_num2label = {i["index"] : i["name"] for i in aal3_num2label["labels"]}

aal3_num2label

### Brodmann Areas — Mai–Majtanik Atlas

Brodmann areas are the classical cytoarchitectonic map of the human cortex, originally
defined by Korbinian Brodmann in 1909 from histological sections and still widely used
to communicate the location of activations in the neuroscience literature. The
Mai–Majtanik atlas <cite data-cite="Mai2017">(Mai & Majtanik, 2017)</cite> provides
these areas as a digitised volumetric label map registered to **MNI152** space, making
it directly compatible with the Cedalion head models. Areas such as BA44/45
(Broca's area) or BA17 (primary visual cortex) are convenient reference landmarks when
linking fNIRS results to the broader neuroimaging literature.

In [ ]:
brodmann_voxel_label_niftii, brodmann_labels_json = cedalion.data.get_atlas_files("brodmann")

# dictionary to map numeric voxel labels in nifti to string labels
with brodmann_labels_json.open("r") as fin:
    brodmann_num2label = json.load(fin)
    brodmann_num2label = {i["index"] : i["name"] for i in brodmann_num2label["labels"]}

brodmann_num2label

## Brain Vertex Coordinates Before Atlas Assignment

The `brain.vertices` xarray `DataArray` already carries several vertex coordinates set
when the head model was built from FreeSurfer outputs:

| Coordinate | Meaning |
|---|---|
| `parcel` | Schaefer2018 parcel label <cite data-cite="Schaefer2018">(Schaefer et al., 2018)</cite> |
| `fsaverage_vertex` | Corresponding vertex index in the FreeSurfer `fsaverage` template |
| `mni152_r/a/s` | MNI152 RAS coordinates of the vertex in mm |

Calling `assign_parcels_via_mni_coords` will **add** a new named coordinate to this
`DataArray` without removing or modifying any existing one. Multiple atlas labels can
therefore coexist on the same vertex, which is what we exploit below.

In [ ]:
colin_ijk.brain.vertices

## Assigning Atlas Labels to Brain Vertices

`TwoSurfaceHeadModel.assign_parcels_via_mni_coords` transfers labels from a volumetric
NIfTI atlas to the brain surface by exploiting the MNI152 coordinates shared by both.
The algorithm works as follows:

1. **Coordinate alignment**: The NIfTI affine transform is used to compute the MNI
   coordinate of every voxel centre. If the atlas is in MNI305 space
   (`voxel_label_crs='mni305'`), the pre-computed affine
   `cedalion.dot.utils.mni305_to_mni152` is applied first to bring voxel coordinates
   into the same MNI152 frame as the head model vertices.

2. **Nearest-labelled-voxel search**: A KD-tree is built from the voxel centres. For
   each brain surface vertex, all voxels within a ball of radius `mni_eps` mm in MNI152
   space are retrieved. Background voxels (label 0 or whichever integer maps to
   `background_label`) are excluded, and the closest *labelled* voxel is selected.

3. **Label assignment**: The integer label of the winning voxel is translated to a
   string via `label_mapping` and stored as a new vertex coordinate. If no labelled
   voxel falls within `mni_eps` mm, the vertex receives `background_label`.

**Key parameters:**

| Parameter | Effect |
|---|---|
| `coordinate_label` | Name of the new vertex coordinate (e.g. `'parcel_aal3'`) |
| `label_mapping` | `{int: str}` dict mapping numeric voxel labels to region names |
| `voxel_label_niftii` | Path to the NIfTI atlas file |
| `voxel_label_crs` | MNI variant of the NIfTI: `'mni152'` (default) or `'mni305'` |
| `mni_eps` | Search radius in mm (default `5`). Increase if many vertices receive `'Background'` |

Each call returns a **new** head model object; the original is not modified
(immutable-style API). We chain two calls to assign both atlases in sequence.

Assigning AAL3 and Brodmann labels to the Colin27 head:

In [ ]:
# Colin27

colin_ijk_labeled = colin_ijk.assign_parcels_via_mni_coords(
    coordinate_label="parcel_aal3",
    label_mapping=aal3_num2label,
    voxel_label_niftii=aal3_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)

colin_ijk_labeled = colin_ijk_labeled.assign_parcels_via_mni_coords(
    coordinate_label="parcel_brodmann",
    label_mapping=brodmann_num2label,
    voxel_label_niftii=brodmann_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)


colin_ijk_labeled.brain.vertices

Because ICBM152 and Colin27 share the same MNI152 coordinate frame, the *identical*
NIfTI atlas files and label dictionaries work unchanged for both head models. This
is the practical benefit of the shared coordinate system: you write the atlas assignment
code once and it applies to any head model that has MNI152 vertex coordinates.

Assigning AAL3 and Brodmann labels to the ICBM152 head:

In [ ]:
# ICBM-152

icbm_ijk_labeled = icbm_ijk.assign_parcels_via_mni_coords(
    coordinate_label="parcel_aal3",
    label_mapping=aal3_num2label,
    voxel_label_niftii=aal3_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)

icbm_ijk_labeled = icbm_ijk_labeled.assign_parcels_via_mni_coords(
    coordinate_label="parcel_brodmann",
    label_mapping=brodmann_num2label,
    voxel_label_niftii=brodmann_voxel_label_niftii,
    voxel_label_crs="mni152",
    mni_eps=5
)


icbm_ijk_labeled.brain.vertices

## Plotting Parcellation Schemes

Cedalion's surface plotting functions accept a list of per-vertex colors (RGB tuples
or matplotlib color specs), one entry per vertex in vertex order. The helper
`cedalion.vis.anatomy.get_vertex_colors_from_coord` builds this list from a named
vertex coordinate and a `color_mapping` that translates string labels to colors.

Three `color_mapping` modes are supported:

| `color_mapping` value | Behaviour |
|---|---|
| `dict` (label → color) | Each label is looked up; vertices not in the dict get `default_color` |
| `None` | A random but deterministic color is generated for every unique label |
| A single color string (e.g. `'r'`) | Every vertex whose label is in `labels` gets that color; the rest get `default_color` |

We first demonstrate with the **Schaefer2018** parcellation that is bundled with the
head model and has an official color map:

In [ ]:
# example: Schaefer parcel colors
schaefer_color_dict = cedalion.data.get_colin27_headmodel_files().load_parcel_colors()
display(schaefer_color_dict)

`get_vertex_colors_from_coord` iterates over every brain surface vertex, reads its
label from the named coordinate, and returns the corresponding color from the mapping.
Vertices whose label is not found in the mapping receive `default_color` (grey by
default). The result is a list in the same vertex order as `brain.vertices`, ready to
pass directly to any of the `cedalion.vis` plotting functions.

In [ ]:
vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel", color_mapping=schaefer_color_dict)
print(f"The brain surface has {colin_ijk_labeled.brain.nvertices} vertices.")
print(f"The list vertex_colors has {len(vertex_colors)} entries. These are the first entries:")
vertex_colors[:4]

`plot_brain_views_grid` renders the brain surface from five standard viewpoints
(superior, left, right, anterior, posterior) in a single figure.
We pass the **inflated** surface here — which has the same vertex order as the pial
surface — so sulcal cortex is unfolded and every parcel is visible at once.

The Schaefer2018 parcellation on the inflated Colin27 cortex:

In [ ]:
plot_brain_views_grid(colin_inflated, vertex_colors, reset_camera=True)

### AAL3 Labels on Colin27

We now visualize the freshly assigned AAL3 labels. Because AAL3 does not ship with a
canonical per-region color map, we pass `color_mapping=None` to let
`get_vertex_colors_from_coord` generate colors automatically. The algorithm uses a
deterministic golden-ratio hue spacing so colors are consistent across calls.

We show both the **pial** (folded) and the **inflated** surface. The inflated view
makes buried sulcal cortex visible and allows you to judge where parcel boundaries
fall relative to major anatomical landmarks. Notice that compared to the Schaefer2018
map above, some AAL3 borders appear slightly less crisp — this is the expected
consequence of the volumetric-to-surface transfer (discussed in detail at the end
of this notebook).

In [ ]:
vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel_aal3", color_mapping=None, default_color="magenta")

plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(colin_inflated, vertex_colors, reset_camera=True)

### Brodmann Labels on Colin27

The same workflow applied to the Brodmann atlas. Brodmann areas are fewer and larger
than AAL3 regions. On the inflated surface the classical areas are clearly recognisable:
BA4/6 along the central sulcus (motor/premotor), BA17/18/19 in the occipital lobe
(visual cortex), and BA44/45 in the left inferior frontal gyrus (Broca's area).

In [ ]:
vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel_brodmann", color_mapping=None)

plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(colin_inflated, vertex_colors, reset_camera=True)

### AAL3 Labels on ICBM152

The same atlas applied to the ICBM152 head model. Any differences from the Colin27
result reflect genuine anatomical differences between the two templates (single-subject
average vs. group average) rather than any difference in the atlas or the transfer
procedure.

In [ ]:
vertex_colors = get_vertex_colors_from_coord(icbm_ijk_labeled.brain, "parcel_aal3", color_mapping=None)

plot_brain_views_grid(icbm_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(icbm_inflated, vertex_colors, reset_camera=True)

### Brodmann Labels on ICBM152

In [ ]:
vertex_colors = get_vertex_colors_from_coord(icbm_ijk_labeled.brain, "parcel_brodmann", color_mapping=None)

plot_brain_views_grid(icbm_ijk_labeled.brain, vertex_colors)
plot_brain_views_grid(icbm_inflated, vertex_colors, reset_camera=True)

## Customizing Colors for ROI Highlighting

In practice you will often want to highlight a small set of regions of interest rather
than visualize the entire atlas at once. `get_vertex_colors_from_coord` supports two
convenience patterns for this.

**Pattern 1 — selected parcels with distinct colors:**

Pass a partial `color_mapping` dict containing only the regions you care about. All
other vertices receive `default_color` (grey). Use this when the highlighted regions
should be distinguished from one another by color — for example when comparing two
adjacent AAL3 frontal regions.

In [ ]:
# select only two parcels and specify colors (any matplotlib color spec works)
# parcels not contained in the mapping get the default color
aal3_color_mapping = {"Frontal_Inf_Oper_R" : "r", "Frontal_Inf_Tri_R" : "g"}

vertex_colors = get_vertex_colors_from_coord(colin_ijk_labeled.brain, "parcel_aal3", aal3_color_mapping)
plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)


**Pattern 2 — highlight a set of parcels in a single color:**

Pass a single color string as `color_mapping` and provide a `labels` list. Every
vertex whose coordinate matches a label in the list gets the specified color; all
other vertices get `default_color`. This gives the clearest signal-vs-background
contrast when you want to show, for example, bilateral Broca's area (BA44 + BA45)
against a neutral grey background.

In [ ]:
# specify only a single color, that will be used for all parcels. Then select only a subset of parcels to color.

vertex_colors = get_vertex_colors_from_coord(
    colin_ijk_labeled.brain,
    "parcel_brodmann",
    color_mapping="r",
    labels=["right_BA44", "right_BA45"],
)
plot_brain_views_grid(colin_ijk_labeled.brain, vertex_colors)

## Spot-Check: Verifying Label Assignments at Known MNI Coordinates

As a sanity check we verify that the assigned labels are anatomically plausible at
three well-known MNI152 locations. We find the nearest brain surface vertex in MNI152
space and read off its Brodmann label.

| MNI coordinate | Expected region |
|---|---|
| `[-10, -90, 0]` | Occipital pole / calcarine sulcus (BA17/18) |
| `[-40, -20, 50]` | Left somatomotor cortex (BA4/6) |
| `[ 40,  20, 30]` | Right prefrontal cortex (BA44/45/46) |

In [ ]:
tests = np.asarray([
    [-10, -90, 0],
    [-40, -20, 50],
    [40, 20, 30],
], dtype=float)

vertex_coords = icbm_ijk_labeled.get_brain_mni152_coords().pint.dequantify().values
vertex_labels = icbm_ijk_labeled.brain.vertices.coords["parcel_brodmann"].values
_, nearest = KDTree(vertex_coords).query(tests)

for mni, vertex_idx in zip(tests, nearest):
    print(f"MNI {mni.tolist()} -> nearest surface label {vertex_labels[vertex_idx]}")

## Parcel-Level Summary Tables

`TwoSurfaceHeadModel.parcel_summary_from_vertex_coordinate` converts the per-vertex
atlas labels into a compact table with **one row per Schaefer2018 parcel** (the
parcellation bundled with the head model). For each Schaefer parcel it reports:

| Column | Meaning |
|---|---|
| `atlas_label` | Dominant atlas region covering that parcel (plurality vote over vertices) |
| `matching_vertices` | Number of vertices whose atlas label equals `atlas_label` |
| `parcel_vertices` | Total number of vertices in the Schaefer parcel |
| `fraction_of_parcel` | Coverage fraction (`matching_vertices / parcel_vertices`) |
| `mni152_r/a/s` | MNI152 centroid of a representative vertex from that parcel |

Medial-wall and background parcels are excluded by default because they do not
correspond to cortical tissue. The function returns a `pandas.DataFrame` that can be
saved with `.to_csv(..., sep='\t')` for use in downstream analyses or supplementary
tables in manuscripts.

In [ ]:
colin_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_aal3", "colin27")

In [ ]:
colin_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_brodmann", "colin27")

In [ ]:
icbm_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_aal3", "icbm152")

In [ ]:
icbm_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_brodmann", "icbm152")

### Optional: Save Summary Tables

The summary tables above are ordinary `pandas.DataFrame` objects. To write them
to disk, set `SAVE_TABLES = True` in the next cell. Files are saved in the
`atlas_parcel_summaries/` folder relative to the notebook kernel's current
working directory.


In [ ]:
from pathlib import Path

SAVE_TABLES = False
OUTPUT_DIR = Path("atlas_parcel_summaries")

if SAVE_TABLES:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    summaries = {
        "aal3_parcel_summary_colin27.tsv": colin_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_aal3", "colin27"),
        "brodmann_parcel_summary_colin27.tsv": colin_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_brodmann", "colin27"),
        "aal3_parcel_summary_icbm152.tsv": icbm_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_aal3", "icbm152"),
        "brodmann_parcel_summary_icbm152.tsv": icbm_ijk_labeled.parcel_summary_from_vertex_coordinate("parcel_brodmann", "icbm152"),
    }
    for filename, summary in summaries.items():
        summary.to_csv(OUTPUT_DIR / filename, sep="\t", index=False)
    print(f"Saved {len(summaries)} summary tables to {OUTPUT_DIR.resolve()}")


## Limitations of the Volumetric MNI Approach

### Why parcel borders appear fuzzy on the inflated surface

The volumetric-to-surface label transfer is inherently approximate. The label of a
brain surface vertex is determined by the *nearest labelled voxel* in 3-D MNI152 space.
Because sulcal folds can bring two cortically distant regions to within a few
millimetres of each other in 3-D, vertices near a sulcal wall may be assigned the
label of the anatomically adjacent — but cortically distant — region instead of their
own.

This artefact is most visible on the **inflated surface**: inflation changes vertex
positions in 3-D while leaving labels fixed. Vertices that sit at the fundus (bottom)
of a deep sulcus can therefore show "bleeding" — incorrect labels — at the inflated
parcel border, even though the label assignment on the folded pial surface appears
correct.

The `mni_eps` search radius controls the trade-off: a *smaller* radius reduces
cross-sulcal contamination but increases the number of vertices that receive
`'Background'` (no labelled voxel found within reach). The default of 5 mm is a
practical compromise for atlases with 1 mm isotropic voxels.

### When to use FreeSurfer-based labeling instead

For applications where parcel boundary precision matters — for example when comparing
fine-grained DOT image reconstruction results to specific cortical regions, or when
building a classifier that relies on exact parcel membership — **surface-native
labeling via FreeSurfer** is the more accurate alternative.

FreeSurfer <cite data-cite="Fischl2012">(Fischl, 2012)</cite> assigns parcels directly
on the cortical surface mesh by registering each vertex to the `fsaverage` template,
where parcel boundaries are defined as vertex-level annotations. Because the boundary
is expressed in the same surface space as the vertex, no voxel-lookup approximation is
introduced and region borders are exact, even on the inflated surface.

Cedalion provides a complete FreeSurfer-based pipeline for individualized head models
in
[43b_individualized_head_models.ipynb](43b_individualized_head_models.ipynb), which
walks through FreeSurfer cortical reconstruction, registration to `fsaverage`, and
surface-native parcel annotation. The standard Colin27 and ICBM152 models were built
with exactly this pipeline; their bundled `parcel` coordinate (Schaefer2018) is a
surface-native annotation — which is why Schaefer borders look sharper than AAL3 or
Brodmann borders on the same inflated surface.

**Practical guidance:**

| Use case | Recommended approach |
|---|---|
| Quick atlas comparison, ROI definitions, group-level reporting | Volumetric MNI lookup (this notebook) |
| High-precision boundary analysis, individual MRI available | FreeSurfer surface-native labeling |
| Bridging a new atlas to an existing surface-native parcellation | Volumetric MNI lookup, compare overlap with Schaefer via `parcel_summary_from_vertex_coordinate` |

## References

In [ ]:
cedalion.bib.dump_to_notebook()